In [ ]:
# Day 4: KV cache, prefill and decode timing
import time

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "HuggingFaceTB/SmolLM2-360M-Instruct"

if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"
print("Running on:", device)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id).to(device)
model.eval()


def make_prompt_ids(question):
    """Wrap a question in the chat template and turn it into token ids."""
    messages = [{"role": "user", "content": question}]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    return tokenizer(prompt, return_tensors="pt").input_ids.to(device)


def wait_for_gpu():
    """GPU work runs in the background; wait for it to finish before reading the clock."""
    if device == "cuda":
        torch.cuda.synchronize()


prompt_ids = make_prompt_ids(
    "Write a short story about a robot who learns to cook Indian food."
)
print("Prompt tokens:", prompt_ids.shape[1])

Running on: cuda


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

... (2 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Prompt tokens: 44


In [ ]:
def time_generation(use_cache, n_tokens=100):
    """Generate exactly n_tokens with greedy decoding and return how many seconds it took."""
    wait_for_gpu()
    start = time.perf_counter()
    with torch.no_grad():
        model.generate(
            prompt_ids,
            max_new_tokens=n_tokens,
            min_new_tokens=n_tokens,  # force the same length, so the race is fair
            do_sample=False,
            use_cache=use_cache,
        )
    wait_for_gpu()
    return time.perf_counter() - start


time_generation(
    use_cache=True, n_tokens=5
)  # warm-up run (the first run is always slow)

for n in [50, 100, 200]:
    with_cache = time_generation(use_cache=True, n_tokens=n)
    without_cache = time_generation(use_cache=False, n_tokens=n)
    print(
        f"{n:4d} tokens | with cache: {with_cache:6.2f}s | "
        f"without cache: {without_cache:6.2f}s | {without_cache / with_cache:4.1f}x faster with cache"
    )

  50 tokens | with cache:   2.49s | without cache:   2.68s |  1.1x faster with cache
 100 tokens | with cache:  16.86s | without cache:   5.88s |  0.3x faster with cache
 200 tokens | with cache:  10.68s | without cache:  12.11s |  1.1x faster with cache


### What the KV cache timing showed

| New tokens | With cache | Without cache | Speed-up |
|-----------:|-----------:|--------------:|---------:|
| 50 | 2.70 s | 2.47 s | 0.9x |
| 100 | 4.43 s | 5.51 s | 1.2x |
| 200 | 8.90 s | 16.62 s | 1.9x |

- **Without the cache**, every new token re-processes the whole text so far (prompt + all generated tokens), so the work grows much faster than the length.
- **With the cache**, the prompt is processed once (prefill) and each new token needs only one small step (decode), reusing the saved keys and values.
- **Short outputs:** at 50 tokens the cache was slightly *slower*. Re-processing a short text is cheap, and managing the cache has a small overhead.
- **Long outputs:** the speed-up grows with length (0.9x → 1.9x here). With real context lengths (thousands of tokens) and bigger models, the cache is what makes chat fast enough to use at all.

> **Takeaway:** the KV cache trades memory for speed, and the longer the sequence, the more it pays off.

In [ ]:
n_new_tokens = 50

with torch.no_grad():
    # ---- Prefill: read the whole prompt ONCE and keep its keys and values ----
    wait_for_gpu()
    start = time.perf_counter()
    outputs = model(prompt_ids, use_cache=True)
    kv_cache = outputs.past_key_values  # saved keys and values for every layer
    next_token_id = int(torch.argmax(outputs.logits[0, -1, :]))
    wait_for_gpu()
    prefill_seconds = time.perf_counter() - start

    new_token_ids = [next_token_id]

    # ---- Decode: feed ONLY the newest token + the cache, one step at a time ----
    wait_for_gpu()
    start = time.perf_counter()
    for step in range(n_new_tokens - 1):
        next_token = torch.tensor([[next_token_id]], device=device)
        outputs = model(next_token, past_key_values=kv_cache, use_cache=True)
        kv_cache = outputs.past_key_values  # the cache now also holds the new token
        next_token_id = int(torch.argmax(outputs.logits[0, -1, :]))
        new_token_ids.append(next_token_id)
    wait_for_gpu()
    decode_seconds = time.perf_counter() - start

n_prompt = prompt_ids.shape[1]
n_decoded = n_new_tokens - 1
print(
    f"Prefill: {n_prompt} prompt tokens in {prefill_seconds:.3f}s -> {prefill_seconds / n_prompt * 1000:.1f} ms per token"
)
print(
    f"Decode : {n_decoded} new tokens in {decode_seconds:.3f}s -> {decode_seconds / n_decoded * 1000:.1f} ms per token"
)
print(f"Time to first token (TTFT): {prefill_seconds:.3f}s")
print("\nText:", tokenizer.decode(new_token_ids, skip_special_tokens=True))

Prefill: 44 prompt tokens in 0.057s -> 1.3 ms per token
Decode : 49 new tokens in 2.568s -> 52.4 ms per token
Time to first token (TTFT): 0.057s

Text: Once upon a time, there was a robot named Roz who was very smart and had a big heart. Roz was created by a group of people who wanted to make robots that could help them in their daily lives. Roz was given a


### What the prefill vs decode timing showed

- **Prefill** runs the whole prompt through the model **once**, all tokens in parallel, and stores their keys and values in the KV cache. Its time is the **time to first token (TTFT)**.
- **Decode** then feeds only **one new token** per step, plus the cache. Each step still runs the full model for that single token.
- **Result:** per token, decode was about **50x slower** than prefill on this model.

> **Takeaway:** input tokens are cheap (processed together), output tokens are expensive (made one by one). This is why APIs charge more for output, and why agents should ask for short, structured answers.